# CKAN を検索して地図に表示する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/u-kitazawa/rhinestone/blob/develop/showcase/02_ckan_search_to_map.ipynb)

この Showcase は、G空間情報センターの公開 CKAN で「河川」を検索し、明示的に広告された直接読込可能なベクター配布物を Rhinestone で解決してから、利用者所有の pyogrio / GeoPandas で地図に表示します。

```text
configure(pyogrio) -> search -> resolve -> vector Resource
    -> open -> GeoDataFrame -> Folium map
```

Rhinestone は検索・Resource の解決・Runtime への委譲を担います。データ読込後の表示や解析は downstream library の責務です。結果と公開状態は Provider 側で変わります。

## Setup

Colab ではこのセルを一度実行します。Rhinestone Core に GIS / Notebook 依存を追加するものではありません。

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "git+https://github.com/u-kitazawa/rhinestone.git@develop",
        "pyogrio==0.13.0",
        "geopandas==1.1.4",
        "folium==0.20.0",
    ],
    check=True,
)

## 検索する

ここでは単一 Provider を構成します。複数 Provider を構成したとき、Rhinestone は Provider 横断の関連度順位を作りません。

In [ ]:
import pyogrio

from rhinestone import configure, sources

app = configure(
    sources=(sources.GEOSPATIAL_JP,),
    dependencies={"pyogrio": pyogrio},
)
results = app.search(text="河川", limit=20)

## 明示的なベクター配布物を解決する

検索結果を通常の解決パイプラインへ渡し、正規化済みの `Resource.format` が pyogrio 対象の明示的なベクター形式で、archive ではない最初の配布物を開きます。URL suffix や archive の中身から形式を推測しません。

In [ ]:
PYOGRIO_VECTOR_FORMATS = {
    "shapefile",
    "geojson",
    "gpkg",
    "flatgeobuf",
    "gml",
    "kml",
    "citygml",
}
resources = (app.resolve(result) for result in results)
resource = next(
    (
        item
        for item in resources
        if item.format in PYOGRIO_VECTOR_FORMATS and item.access_plan.archive is None
    ),
    None,
)
if resource is None:
    raise RuntimeError(
        "The CKAN search returned no explicitly advertised direct vector resources. "
        "Change the search term; this notebook will not infer a format or archive member."
    )


frame = resource.open("pyogrio")

## 利用者所有の Runtime で開き、地図として表示する

pyogrio は利用者が導入・登録する Execution Runtime です。Rhinestone は選択済みの URI と、Source が確定した属性だけを Runtime に渡します。開いた GeoDataFrame は Folium のベクターレイヤーにし、選択したデータの範囲へ地図を合わせます。

In [ ]:
import folium

if frame.crs is None:
    raise RuntimeError(
        "The selected Resource has no CRS; it cannot be placed on a web map."
    )

GSI_STANDARD_TILES = "https://cyberjapandata.gsi.go.jp/xyz/std/{z}/{x}/{y}.png"
map_frame = frame.to_crs(epsg=4326)[["geometry"]]
west, south, east, north = map_frame.total_bounds
map_view = folium.Map(tiles=None)
folium.TileLayer(
    tiles=GSI_STANDARD_TILES,
    attr="GSI Maps",
).add_to(map_view)
folium.GeoJson(
    data=map_frame.__geo_interface__,
).add_to(map_view)
map_view.fit_bounds([[south, west], [north, east]])

map_view

## 境界

この Notebook は検索、解決、AccessPlan、Runtime への委譲を示します。操作できる地図と地理院タイルの背景タイルは pyogrio / GeoPandas / Folium の機能であり、Rhinestone は GIS 解析、形式変換、背景地図、archive の展開を実装しません。